In [ ]:
from pathlib import Path
import os

SRC_DIR = Path.cwd()          # current working directory
PROJECT_DIR = SRC_DIR.parent  # parent folder containing src, dataset, visualizations

DATASET_DIR = PROJECT_DIR / "dataset"
VISUALIZATIONS_DIR = PROJECT_DIR / "visualizations"

DATASET_DIR.mkdir(parents=True, exist_ok=True)
VISUALIZATIONS_DIR.mkdir(parents=True, exist_ok=True)

COUNTRIES_FILE = DATASET_DIR / "wb_countries.json"
ECON_FILE = DATASET_DIR / "economic_indicators_raw.json"
EDU_FILE = DATASET_DIR / "education_indicators_raw.json"

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

print("OpenMP environment variables set.")
print("SRC_DIR =", SRC_DIR)
print("PROJECT_DIR =", PROJECT_DIR)

NameError: name '__file__' is not defined

In [ ]:
import sys
import json
import time
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv
from pymongo import MongoClient, ASCENDING
from pymongo.errors import BulkWriteError
import psycopg2
from psycopg2.extras import execute_values
from sqlalchemy import create_engine, text
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
sns.set_style("whitegrid")
sns.set_palette("Set2")
DATASET_DIR.mkdir(parents=True, exist_ok=True)
VISUALIZATIONS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python  : {sys.version.split()[0]}")
print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")
print("All imports successful.")

In [ ]:
# Load credentials from .env file
load_dotenv()

# MongoDB
MONGO_URI     = os.getenv("MONGO_URI")
MONGO_DB_NAME = os.getenv("MONGO_DB_NAME", "socioeconomic_analytics")

# PostgreSQL
PG_HOST     = os.getenv("PG_HOST", "localhost")
PG_PORT     = int(os.getenv("PG_PORT", 5432))
PG_USER     = os.getenv("PG_USER", "postgres")
PG_PASSWORD = os.getenv("PG_PASSWORD", "arjun")
PG_DB       = os.getenv("PG_DB", "socioeconomic_analytics")

# World Bank API
WB_BASE_URL = "https://api.worldbank.org/v2"



ECONOMIC_INDICATORS = {
    "NY.GDP.PCAP.CD":     "gdp_per_capita",
    "SP.DYN.LE00.IN":     "life_expectancy",
    "SP.POP.TOTL":        "population",
    "SL.UEM.TOTL.ZS":    "unemployment_rate",
    "BX.KLT.DINV.CD.WD": "fdi_inflows",
}

EDUCATION_INDICATORS = {
    "SE.ADT.LITR.ZS":    "literacy_rate",
    "SE.PRM.ENRR":       "primary_enrollment",
    "SE.SEC.ENRR":       "secondary_enrollment",
    "SE.TER.ENRR":       "tertiary_enrollment",
    "SE.XPD.TOTL.GD.ZS": "edu_expenditure_pct_gdp",
}

START_YEAR = 2000
END_YEAR   = 2022

G7    = ["USA", "GBR", "DEU", "FRA", "JPN", "ITA", "CAN"]
BRICS = ["BRA", "RUS", "IND", "CHN", "ZAF"]
FOCUS = G7 + BRICS

INCOME_ORDER = [
    "Low income", "Lower middle income",
    "Upper middle income", "High income",
]

print("Configuration loaded from .env")
print(f"  MongoDB DB  : {MONGO_DB_NAME}")
print(f"  PG host     : {PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"  Dataset dir : {DATASET_DIR}/")
local = {
    "countries"  : os.path.exists(COUNTRIES_FILE),
    "economic"   : os.path.exists(ECON_FILE),
    "education"  : os.path.exists(EDU_FILE),
}
print(f"  Local files : {local}")

In [ ]:
# Section 2 - Database Initialisation

def create_pg_database():
    """Create the PostgreSQL database if it does not already exist."""
    conn = psycopg2.connect(
        host=PG_HOST, port=PG_PORT,
        user=PG_USER, password=PG_PASSWORD,
        database="postgres",
    )
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (PG_DB,))
    if not cur.fetchone():
        cur.execute(f"CREATE DATABASE {PG_DB}")
        print(f"Database '{PG_DB}' created.")
    else:
        print(f"Database '{PG_DB}' already exists.")
    cur.close()
    conn.close()


create_pg_database()

In [ ]:
CREATE_TABLES_SQL = """
CREATE TABLE IF NOT EXISTS countries (
    country_code  VARCHAR(3)   PRIMARY KEY,
    country_name  VARCHAR(150) NOT NULL,
    region        VARCHAR(80),
    income_group  VARCHAR(80),
    capital_city  VARCHAR(100),
    longitude     DOUBLE PRECISION,
    latitude      DOUBLE PRECISION
);

CREATE TABLE IF NOT EXISTS raw_economic_records (
    id             SERIAL PRIMARY KEY,
    country_code   VARCHAR(3),
    country_name   VARCHAR(150),
    indicator_code VARCHAR(50),
    indicator_name VARCHAR(200),
    year           INTEGER,
    value          DOUBLE PRECISION,
    fetched_at     TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS raw_education_records (
    id             SERIAL PRIMARY KEY,
    country_code   VARCHAR(3),
    country_name   VARCHAR(150),
    indicator_code VARCHAR(50),
    indicator_name VARCHAR(200),
    year           INTEGER,
    value          DOUBLE PRECISION,
    fetched_at     TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS economic_indicators (
    id                SERIAL PRIMARY KEY,
    country_code      VARCHAR(3),
    country_name      VARCHAR(150),
    year              INTEGER,
    gdp_per_capita    DOUBLE PRECISION,
    life_expectancy   DOUBLE PRECISION,
    population        BIGINT,
    unemployment_rate DOUBLE PRECISION,
    fdi_inflows       DOUBLE PRECISION,
    UNIQUE (country_code, year)
);

CREATE TABLE IF NOT EXISTS education_indicators (
    id                      SERIAL PRIMARY KEY,
    country_code            VARCHAR(3),
    country_name            VARCHAR(150),
    year                    INTEGER,
    literacy_rate           DOUBLE PRECISION,
    primary_enrollment      DOUBLE PRECISION,
    secondary_enrollment    DOUBLE PRECISION,
    tertiary_enrollment     DOUBLE PRECISION,
    edu_expenditure_pct_gdp DOUBLE PRECISION,
    UNIQUE (country_code, year)
);

CREATE TABLE IF NOT EXISTS merged_development_data (
    id                      SERIAL PRIMARY KEY,
    country_code            VARCHAR(3),
    country_name            VARCHAR(150),
    year                    INTEGER,
    region                  VARCHAR(80),
    income_group            VARCHAR(80),
    gdp_per_capita          DOUBLE PRECISION,
    life_expectancy         DOUBLE PRECISION,
    population              BIGINT,
    unemployment_rate       DOUBLE PRECISION,
    fdi_inflows             DOUBLE PRECISION,
    literacy_rate           DOUBLE PRECISION,
    primary_enrollment      DOUBLE PRECISION,
    secondary_enrollment    DOUBLE PRECISION,
    tertiary_enrollment     DOUBLE PRECISION,
    edu_expenditure_pct_gdp DOUBLE PRECISION,
    log_gdp_per_capita      DOUBLE PRECISION,
    UNIQUE (country_code, year)
);
"""

pg_engine = create_engine(
    f"postgresql://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)

with pg_engine.connect() as conn:
    conn.execute(text(CREATE_TABLES_SQL))
    conn.commit()
print("PostgreSQL tables created / verified.")

In [ ]:
mongo_client = MongoClient(
    MONGO_URI,
    serverSelectionTimeoutMS=20000
)
mongo_db = mongo_client[MONGO_DB_NAME]

col_raw_econ  = mongo_db["raw_economic_records"]
col_raw_edu   = mongo_db["raw_education_records"]
col_countries = mongo_db["wb_countries"]

# Compound unique indexes to prevent duplicate inserts on re-runs
col_raw_econ.create_index(
    [("country_code", ASCENDING), ("indicator_code", ASCENDING), ("year", ASCENDING)],
    unique=True, name="econ_unique_idx"
)
col_raw_edu.create_index(
    [("country_code", ASCENDING), ("indicator_code", ASCENDING), ("year", ASCENDING)],
    unique=True, name="edu_unique_idx"
)

print(f"MongoDB connected   : {mongo_db.name}")
print(f"Collections present : {mongo_db.list_collection_names()}")

In [ ]:
# Section 3 - Data Ingestion

def fetch_wb_data(indicator_code, start_year=START_YEAR, end_year=END_YEAR,
                  per_page=1000):
    """Fetch paginated World Bank indicator records for all countries."""
    all_records = []
    page        = 1
    total_pages = None

    while total_pages is None or page <= total_pages:
        url    = f"{WB_BASE_URL}/country/all/indicator/{indicator_code}"
        params = {
            "format"   : "json",
            "per_page" : per_page,
            "page"     : page,
            "date"     : f"{start_year}:{end_year}",
        }
        try:
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            payload = resp.json()
        except (requests.RequestException, ValueError) as exc:
            print(f"  Warning: page {page} failed for {indicator_code} - {exc}")
            break

        if len(payload) < 2 or not payload[1]:
            break

        meta, records = payload[0], payload[1]
        all_records.extend(records)
        total_pages = meta.get("pages", 1)
        if page >= total_pages:
            break
        page += 1
        time.sleep(0.25)

    return all_records


def fetch_wb_countries():
    """Fetch country metadata from the World Bank API."""
    url  = f"{WB_BASE_URL}/country"
    resp = requests.get(url, params={"format": "json", "per_page": 500}, timeout=30)
    resp.raise_for_status()
    payload = resp.json()
    return payload[1] if len(payload) > 1 else []


def is_valid_country(iso3):
    """Return True if iso3 is a real 3-alpha country code, not an aggregate."""
    return isinstance(iso3, str) and len(iso3) == 3 and iso3.isalpha()


def parse_wb_records(raw_records):
    """Parse raw World Bank API records into clean dicts; skip aggregates."""
    parsed = []
    for rec in raw_records:
        iso3 = rec.get("countryiso3code", "")
        if not is_valid_country(iso3):
            continue
        parsed.append({
            "country_code"   : iso3,
            "country_name"   : rec["country"]["value"],
            "indicator_code" : rec["indicator"]["id"],
            "indicator_name" : rec["indicator"]["value"],
            "year"           : int(rec["date"]),
            "value"          : (
                float(rec["value"]) if rec.get("value") is not None else None
            ),
        })
    return parsed


def mongo_bulk_insert(collection, docs):
    """Insert documents into MongoDB, silently skipping duplicates."""
    inserted = 0
    for start in range(0, len(docs), 500):
        batch = docs[start : start + 500]
        try:
            result   = collection.insert_many(batch, ordered=False)
            inserted += len(result.inserted_ids)
        except BulkWriteError as bwe:
            inserted += bwe.details.get("nInserted", 0)
    return inserted


print("API helper functions defined.")

In [ ]:
country_docs   = []
countries_rows = []

if os.path.exists(COUNTRIES_FILE):
    print(f"Loading country metadata from local file: {COUNTRIES_FILE}")
    with open(COUNTRIES_FILE, "r", encoding="utf-8") as fh:
        raw_countries = json.load(fh)
    print(f"  {len(raw_countries):,} country records loaded.")
else:
    print("Fetching World Bank country metadata from API ...")
    raw_countries = fetch_wb_countries()
  
    with open(COUNTRIES_FILE, "w", encoding="utf-8") as fh:
        json.dump([c for c in raw_countries if is_valid_country(c.get("id", ""))], fh,
                  ensure_ascii=False, indent=2)
    print(f"  Saved to {COUNTRIES_FILE}")

for c in raw_countries:
    iso3 = c.get("id", "")
    if not is_valid_country(iso3):
        continue
    lon = c.get("longitude")
    lat = c.get("latitude")
    doc = {
        "country_code" : iso3,
        "country_name" : c.get("name", ""),
        "iso2_code"    : c.get("iso2Code", ""),
        "region"       : c.get("region",      {}).get("value"),
        "income_group" : c.get("incomeLevel", {}).get("value"),
        "capital_city" : c.get("capitalCity", ""),
        "longitude"    : float(lon) if lon else None,
        "latitude"     : float(lat) if lat else None,
    }
    country_docs.append(doc)
    countries_rows.append({k: doc[k] for k in doc if k != "iso2_code"})

# MongoDB
col_countries.drop()
col_countries.insert_many(country_docs)
print(f"  MongoDB   : {len(country_docs)} country documents stored.")

# PostgreSQL - upsert to handle re-runs
df_countries = pd.DataFrame(countries_rows)
upsert_sql = text("""
    INSERT INTO countries (country_code, country_name, region, income_group,
                           capital_city, longitude, latitude)
    VALUES (:country_code, :country_name, :region, :income_group,
            :capital_city, :longitude, :latitude)
    ON CONFLICT (country_code) DO UPDATE
        SET country_name = EXCLUDED.country_name,
            region       = EXCLUDED.region,
            income_group = EXCLUDED.income_group,
            capital_city = EXCLUDED.capital_city,
            longitude    = EXCLUDED.longitude,
            latitude     = EXCLUDED.latitude
""")
with pg_engine.connect() as conn:
    conn.execute(upsert_sql, df_countries.to_dict(orient="records"))
    conn.commit()
print(f"  PostgreSQL: {len(df_countries)} countries upserted.")
df_countries.head(3)

In [ ]:
print("=" * 62)
print("Loading Economic Indicators ...")
print("=" * 62)

all_econ_docs = []

if os.path.exists(ECON_FILE):
    print(f"Found local file: {ECON_FILE}")
    with open(ECON_FILE, "r", encoding="utf-8") as fh:
        all_econ_docs = json.load(fh)
    print(f"  {len(all_econ_docs):,} records loaded from dataset/")
else:
    econ_already = col_raw_econ.count_documents({})
    if econ_already > 5000:
        print(f"Economic data already in MongoDB ({econ_already:,} docs). Skipping fetch.")
        all_econ_docs = list(col_raw_econ.find({}, {"_id": 0}))
    else:
        print("Fetching from World Bank API (run download_data.py to cache locally) ...")
        for code, col_name in ECONOMIC_INDICATORS.items():
            print(f"  Fetching {col_name} ({code}) ...")
            raw = fetch_wb_data(code)
            parsed = parse_wb_records(raw)
            all_econ_docs.extend(parsed)
            print(f"    {len(parsed):,} valid records")
        
        with open(ECON_FILE, "w", encoding="utf-8") as fh:
            json.dump(all_econ_docs, fh, ensure_ascii=False, indent=2)
        print(f"  Saved to {ECON_FILE}")

n = mongo_bulk_insert(col_raw_econ, all_econ_docs)
print(f"\nMongoDB : {n:,} records stored / skipped duplicates.")

df_econ_raw_pg = pd.DataFrame(all_econ_docs)
if "_id" in df_econ_raw_pg.columns:
    df_econ_raw_pg = df_econ_raw_pg.drop(columns=["_id"])
with pg_engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE raw_economic_records RESTART IDENTITY"))
    conn.commit()
df_econ_raw_pg.to_sql(
    "raw_economic_records", pg_engine,
    if_exists="append", index=False, method="multi", chunksize=1000
)
print(f"PostgreSQL raw_economic_records: {len(df_econ_raw_pg):,} rows.")
print(f"\nTotal economic docs in MongoDB: {col_raw_econ.count_documents({}):,}")

In [ ]:
print("=" * 62)
print("Loading Education Indicators ...")
print("=" * 62)

all_edu_docs = []

if os.path.exists(EDU_FILE):
    print(f"Found local file: {EDU_FILE}")
    with open(EDU_FILE, "r", encoding="utf-8") as fh:
        all_edu_docs = json.load(fh)
    print(f"  {len(all_edu_docs):,} records loaded from dataset/")
else:
    edu_already = col_raw_edu.count_documents({})
    if edu_already > 5000:
        print(f"Education data already in MongoDB ({edu_already:,} docs). Skipping fetch.")
        all_edu_docs = list(col_raw_edu.find({}, {"_id": 0}))
    else:
        print("Fetching from World Bank API (run download_data.py to cache locally) ...")
        for code, col_name in EDUCATION_INDICATORS.items():
            print(f"  Fetching {col_name} ({code}) ...")
            raw    = fetch_wb_data(code)
            parsed = parse_wb_records(raw)
            all_edu_docs.extend(parsed)
            print(f"    {len(parsed):,} valid records")
        
        with open(EDU_FILE, "w", encoding="utf-8") as fh:
            json.dump(all_edu_docs, fh, ensure_ascii=False, indent=2)
        print(f"  Saved to {EDU_FILE}")

n = mongo_bulk_insert(col_raw_edu, all_edu_docs)
print(f"\nMongoDB : {n:,} records stored / skipped duplicates.")

df_edu_raw_pg = pd.DataFrame(all_edu_docs)
if "_id" in df_edu_raw_pg.columns:
    df_edu_raw_pg = df_edu_raw_pg.drop(columns=["_id"])
with pg_engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE raw_education_records RESTART IDENTITY"))
    conn.commit()
df_edu_raw_pg.to_sql(
    "raw_education_records", pg_engine,
    if_exists="append", index=False, method="multi", chunksize=1000
)
print(f"PostgreSQL raw_education_records: {len(df_edu_raw_pg):,} rows.")
print(f"\nTotal education docs in MongoDB: {col_raw_edu.count_documents({}):,}")

In [ ]:
# Section 4 - Data Retrieval from MongoDB

PROJ = {
    "_id": 0, "country_code": 1, "country_name": 1,
    "indicator_code": 1, "year": 1, "value": 1,
}

print("Retrieving economic records from MongoDB ...")
df_econ_raw = pd.DataFrame(list(col_raw_econ.find({}, PROJ)))
print(f"  Rows: {len(df_econ_raw):,}")

print("Retrieving education records from MongoDB ...")
df_edu_raw = pd.DataFrame(list(col_raw_edu.find({}, PROJ)))
print(f"  Rows: {len(df_edu_raw):,}")

In [ ]:
print("=" * 62)
print("DATA QUALITY ASSESSMENT")
print("=" * 62)

for label, df_q in [("Economic Raw", df_econ_raw), ("Education Raw", df_edu_raw)]:
    print(f"\n--- {label} ---")
    print(f"  Shape      : {df_q.shape}")
    print(f"  Countries  : {df_q['country_code'].nunique()}")
    print(f"  Indicators : {df_q['indicator_code'].nunique()}")
    print(f"  Year range : {df_q['year'].min()} - {df_q['year'].max()}")
    pct = df_q["value"].isna().mean() * 100
    print(f"  Missing %%  : {pct:.1f}%%")

In [ ]:
# Section 5 - Data Cleaning and Transformation

econ_inv = {k: v for k, v in ECONOMIC_INDICATORS.items()}

df_econ_valid = df_econ_raw.dropna(subset=["value"]).copy()
df_econ_valid["indicator_col"] = df_econ_valid["indicator_code"].map(econ_inv)

df_econ_pivot = (
    df_econ_valid
    .pivot_table(
        index=["country_code", "country_name", "year"],
        columns="indicator_col",
        values="value",
        aggfunc="mean",
    )
    .reset_index()
)
df_econ_pivot.columns.name = None

for col in ECONOMIC_INDICATORS.values():
    if col not in df_econ_pivot.columns:
        df_econ_pivot[col] = np.nan

if "population" in df_econ_pivot.columns:
    df_econ_pivot["population"] = df_econ_pivot["population"].apply(
        lambda x: int(x) if pd.notna(x) else None
    )

print(f"Economic pivot shape : {df_econ_pivot.shape}")
df_econ_pivot.head(3)

In [ ]:
edu_inv = {k: v for k, v in EDUCATION_INDICATORS.items()}

df_edu_valid = df_edu_raw.dropna(subset=["value"]).copy()
df_edu_valid["indicator_col"] = df_edu_valid["indicator_code"].map(edu_inv)

df_edu_pivot = (
    df_edu_valid
    .pivot_table(
        index=["country_code", "country_name", "year"],
        columns="indicator_col",
        values="value",
        aggfunc="mean",
    )
    .reset_index()
)
df_edu_pivot.columns.name = None

for col in EDUCATION_INDICATORS.values():
    if col not in df_edu_pivot.columns:
        df_edu_pivot[col] = np.nan

# Clip gross enrollment > 150 (known data artefact)
for col in ["primary_enrollment", "secondary_enrollment", "tertiary_enrollment"]:
    if col in df_edu_pivot.columns:
        df_edu_pivot[col] = df_edu_pivot[col].clip(upper=150)

print(f"Education pivot shape: {df_edu_pivot.shape}")
df_edu_pivot.head(3)

In [ ]:
# Section 6 - Data Integration and Processed Storage

# Outer-merge economic and education datasets on country + year
df_merged = pd.merge(
    df_econ_pivot,
    df_edu_pivot.drop(columns=["country_name"], errors="ignore"),
    on=["country_code", "year"],
    how="outer",
)

# Attach region and income group from country metadata
df_meta   = df_countries[["country_code", "region", "income_group"]].copy()
df_merged = pd.merge(df_merged, df_meta, on="country_code", how="left")

# Keep only actual country rows
df_merged = df_merged[df_merged["country_code"].apply(is_valid_country)].copy()
df_merged.reset_index(drop=True, inplace=True)

# Feature engineering
df_merged["log_gdp_per_capita"] = np.log1p(df_merged["gdp_per_capita"])

print(f"Merged dataset shape : {df_merged.shape}")
print(f"Countries            : {df_merged['country_code'].nunique()}")
print(f"Year range           : {df_merged['year'].min()} - {df_merged['year'].max()}")
print(f"Income groups        : {df_merged['income_group'].nunique()}")
print(f"Regions              : {df_merged['region'].nunique()}")

In [ ]:
# Store economic_indicators
with pg_engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE economic_indicators RESTART IDENTITY"))
    conn.commit()

econ_cols = ["country_code", "country_name", "year",
             "gdp_per_capita", "life_expectancy", "population",
             "unemployment_rate", "fdi_inflows"]
df_econ_pivot[[c for c in econ_cols if c in df_econ_pivot.columns]] \
    .drop_duplicates(subset=["country_code", "year"]) \
    .to_sql(
        "economic_indicators", pg_engine, if_exists="append",
        index=False, method="multi", chunksize=500
    )

# Store education_indicators
with pg_engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE education_indicators RESTART IDENTITY"))
    conn.commit()

edu_cols = ["country_code", "country_name", "year",
            "literacy_rate", "primary_enrollment", "secondary_enrollment",
            "tertiary_enrollment", "edu_expenditure_pct_gdp"]
df_edu_pivot[[c for c in edu_cols if c in df_edu_pivot.columns]] \
    .drop_duplicates(subset=["country_code", "year"]) \
    .to_sql(
        "education_indicators", pg_engine, if_exists="append",
        index=False, method="multi", chunksize=500
    )

# Store merged_development_data
with pg_engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE merged_development_data RESTART IDENTITY"))
    conn.commit()

merged_cols = [
    "country_code", "country_name", "year", "region", "income_group",
    "gdp_per_capita", "life_expectancy", "population",
    "unemployment_rate", "fdi_inflows",
    "literacy_rate", "primary_enrollment", "secondary_enrollment",
    "tertiary_enrollment", "edu_expenditure_pct_gdp",
    "log_gdp_per_capita",
]
df_out = df_merged[[c for c in merged_cols if c in df_merged.columns]].copy()
if "population" in df_out.columns:
    df_out["population"] = df_out["population"].apply(
        lambda x: int(x) if pd.notna(x) and abs(x) < 9e18 else None
    )
df_out.drop_duplicates(subset=["country_code", "year"]).to_sql(
    "merged_development_data", pg_engine, if_exists="append",
    index=False, method="multi", chunksize=500
)
print("Processed data stored to PostgreSQL.")

In [ ]:
tables = [
    "countries", "raw_economic_records", "raw_education_records",
    "economic_indicators", "education_indicators", "merged_development_data",
]
print("Table record counts (PostgreSQL):")
with pg_engine.connect() as conn:
    for t in tables:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
        print(f"  {t:<32} {n:>8,}")

In [ ]:
# Section 7 - Exploratory Data Analysis

with pg_engine.connect() as conn:
    df = pd.read_sql(
        text("SELECT * FROM merged_development_data ORDER BY country_code, year"),
        conn,
    )

print(f"Loaded {len(df):,} rows from PostgreSQL.")
print("\nDescriptive Statistics:")
num_cols = [
    "gdp_per_capita", "life_expectancy", "unemployment_rate",
    "literacy_rate", "secondary_enrollment", "edu_expenditure_pct_gdp",
]
df[num_cols].describe().round(2)

In [ ]:
completeness = df[num_cols].notna().mean().sort_values(ascending=True) * 100

fig, ax = plt.subplots(figsize=(10, 4))
completeness.plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("Data Completeness (%)")
ax.set_title("Data Completeness by Indicator", fontweight="bold")
for i, val in enumerate(completeness):
    ax.text(val + 0.5, i, f"{val:.0f}%", va="center", fontsize=9)
ax.set_xlim(0, 115)
plt.tight_layout()
plt.savefig(VISUALIZATIONS_DIR /"00_data_completeness.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
labels = [
    ("gdp_per_capita",         "GDP per Capita (USD)"),
    ("life_expectancy",         "Life Expectancy (years)"),
    ("unemployment_rate",       "Unemployment Rate (%)"),
    ("literacy_rate",           "Literacy Rate (%)"),
    ("secondary_enrollment",    "Secondary Enrollment (%)"),
    ("edu_expenditure_pct_gdp", "Education Expenditure (% GDP)"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle("Distribution of Key Socioeconomic Indicators",
             fontsize=14, fontweight="bold")

for ax, (col, title) in zip(axes.flat, labels):
    valid = df[col].dropna()
    ax.hist(valid, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
    ax.axvline(valid.median(), color="crimson", linestyle="--",
               linewidth=1.6, label=f"Median: {valid.median():.1f}")
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(VISUALIZATIONS_DIR /"01_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Section 8 - Visualisations
# Chart 1 - Choropleth: GDP per capita (latest available year per country)
df_latest = (
    df.dropna(subset=["gdp_per_capita"])
      .sort_values("year")
      .groupby("country_code", as_index=False)
      .last()
)

fig = px.choropleth(
    df_latest,
    locations="country_code",
    color="gdp_per_capita",
    hover_name="country_name",
    hover_data={"year": True, "gdp_per_capita": ":.0f"},
    color_continuous_scale="Viridis",
    labels={"gdp_per_capita": "GDP per Capita (USD)"},
    title="World GDP per Capita - Latest Available Year",
)
fig.update_layout(coloraxis_colorbar=dict(title="USD"))
fig.write_html(VISUALIZATIONS_DIR /"02_choropleth_gdp.html")
fig.show()

In [ ]:
# Chart 2 - Line: Average GDP per capita by region over time
df_region_trend = (
    df.dropna(subset=["gdp_per_capita", "region"])
      .groupby(["region", "year"], as_index=False)["gdp_per_capita"]
      .mean()
)

fig, ax = plt.subplots(figsize=(13, 6))
for region, grp in df_region_trend.groupby("region"):
    ax.plot(grp["year"], grp["gdp_per_capita"],
            marker="o", markersize=3, linewidth=1.8, label=region)

ax.set_title("Average GDP per Capita by Region (2000-2022)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("GDP per Capita (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(loc="upper left", fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(VISUALIZATIONS_DIR /"03_gdp_region_trend.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 3 - Bubble: GDP vs Life Expectancy sized by Population (2019)
df_bubble = (
    df.dropna(subset=["gdp_per_capita", "life_expectancy",
                      "population", "income_group"])
      .query("year == 2019")
)

fig = px.scatter(
    df_bubble,
    x="gdp_per_capita",
    y="life_expectancy",
    size="population",
    color="income_group",
    hover_name="country_name",
    size_max=60,
    log_x=True,
    labels={
        "gdp_per_capita" : "GDP per Capita (USD, log scale)",
        "life_expectancy": "Life Expectancy (years)",
        "income_group"   : "Income Group",
    },
    title="GDP per Capita vs Life Expectancy (2019) - Bubble size = Population",
)
fig.write_html(VISUALIZATIONS_DIR /"04_bubble_gdp_le.html")
fig.show()

In [ ]:
# Chart 4 - Horizontal bar: Top 20 vs Bottom 20 by GDP per capita (2019)
df_2019 = df.dropna(subset=["gdp_per_capita"]).query("year == 2019")
top20   = df_2019.nlargest(20,  "gdp_per_capita")
bot20   = df_2019.nsmallest(20, "gdp_per_capita")

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, subset, title, color in [
    (axes[0], top20, "Top 20 - GDP per Capita (2019)",    "mediumseagreen"),
    (axes[1], bot20, "Bottom 20 - GDP per Capita (2019)", "tomato"),
]:
    s = subset.sort_values("gdp_per_capita")
    ax.barh(s["country_name"], s["gdp_per_capita"], color=color, edgecolor="white")
    ax.set_xlabel("GDP per Capita (USD)")
    ax.set_title(title, fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
plt.savefig(VISUALIZATIONS_DIR /"05_top_bottom_gdp.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 5 - Correlation heatmap
corr_cols = [
    "gdp_per_capita", "life_expectancy", "unemployment_rate",
    "literacy_rate", "secondary_enrollment", "tertiary_enrollment",
    "edu_expenditure_pct_gdp", "fdi_inflows",
]
corr_matrix = df[corr_cols].dropna().corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="RdYlGn", center=0, linewidths=0.5,
    ax=ax, annot_kws={"size": 9},
)
ax.set_title("Correlation Matrix - Socioeconomic and Education Indicators",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(VISUALIZATIONS_DIR /"06_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 6 - Box plot: GDP per Capita by Income Group
df_box = df.dropna(subset=["gdp_per_capita", "income_group"]).copy()
df_box["income_group"] = pd.Categorical(
    df_box["income_group"], categories=INCOME_ORDER, ordered=True
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(
    data=df_box.dropna(subset=["income_group"]),
    x="income_group", y="gdp_per_capita",
    palette="Set2", order=INCOME_ORDER, ax=ax, showfliers=False,
)
ax.set_title("GDP per Capita Distribution by Income Group",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Income Group")
ax.set_ylabel("GDP per Capita (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
plt.savefig(VISUALIZATIONS_DIR /"07_boxplot_income_gdp.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 7 - Violin: Life Expectancy by Income Group
df_viol = df.dropna(subset=["life_expectancy", "income_group"]).copy()
df_viol["income_group"] = pd.Categorical(
    df_viol["income_group"], categories=INCOME_ORDER, ordered=True
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(
    data=df_viol.dropna(subset=["income_group"]),
    x="income_group", y="life_expectancy",
    palette="Set3", order=INCOME_ORDER, ax=ax, inner="quartile",
)
ax.set_title("Life Expectancy by Income Group", fontsize=13, fontweight="bold")
ax.set_xlabel("Income Group")
ax.set_ylabel("Life Expectancy (years)")
plt.tight_layout()
plt.savefig(VISUALIZATIONS_DIR /"08_violin_life_exp.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 8 - Interactive line: G7 + BRICS GDP trend
df_focus = (
    df[df["country_code"].isin(FOCUS)]
      .dropna(subset=["gdp_per_capita"])
      .sort_values("year")
)
fig = px.line(
    df_focus,
    x="year", y="gdp_per_capita",
    color="country_name",
    markers=True,
    labels={
        "gdp_per_capita": "GDP per Capita (USD)",
        "year"          : "Year",
        "country_name"  : "Country",
    },
    title="GDP per Capita - G7 and BRICS Nations (2000-2022)",
)
fig.update_traces(marker=dict(size=4))
fig.write_html(VISUALIZATIONS_DIR /"09_g7_brics_gdp.html")
fig.show()

In [ ]:
# Chart 9 - Line: Literacy rate trend by income group
df_lit = (
    df.dropna(subset=["literacy_rate", "income_group"])
      .groupby(["income_group", "year"], as_index=False)["literacy_rate"]
      .mean()
)
df_lit["income_group"] = pd.Categorical(
    df_lit["income_group"], categories=INCOME_ORDER, ordered=True
)

fig, ax = plt.subplots(figsize=(13, 6))
for ig, grp in df_lit.dropna(subset=["income_group"]).groupby(
    "income_group", observed=True
):
    ax.plot(grp["year"], grp["literacy_rate"],
            marker="o", markersize=3, linewidth=2, label=ig)
ax.set_title("Average Literacy Rate by Income Group (2000-2022)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Literacy Rate (%)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(VISUALIZATIONS_DIR /"10_literacy_income_trend.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 10 - Scatter: Secondary enrollment vs Log GDP per Capita
df_scat = df.dropna(
    subset=["secondary_enrollment", "log_gdp_per_capita", "income_group"]
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(
    data=df_scat,
    x="log_gdp_per_capita", y="secondary_enrollment",
    hue="income_group", palette="Set2",
    alpha=0.45, s=22, ax=ax, hue_order=INCOME_ORDER,
)
ax.set_xlabel("Log GDP per Capita")
ax.set_ylabel("Secondary School Enrollment (%)")
ax.set_title("Secondary Enrollment vs Log GDP per Capita",
             fontsize=13, fontweight="bold")
ax.legend(title="Income Group", fontsize=8, title_fontsize=9)
plt.tight_layout()
plt.savefig(VISUALIZATIONS_DIR /"11_enrollment_vs_gdp.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 11 - Bar: Education expenditure by region (latest 5-year average)
yr_max   = int(df["year"].max())
df_edu_bar = (
    df.query(f"year >= {yr_max - 4}")
      .dropna(subset=["edu_expenditure_pct_gdp", "region"])
      .groupby("region", as_index=False)["edu_expenditure_pct_gdp"]
      .mean()
      .sort_values("edu_expenditure_pct_gdp", ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=df_edu_bar, x="region", y="edu_expenditure_pct_gdp",
    palette="muted", ax=ax,
)
ax.set_title(f"Avg Education Expenditure by Region ({yr_max - 4}-{yr_max})",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Region")
ax.set_ylabel("Education Expenditure (% of GDP)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(VISUALIZATIONS_DIR /"12_edu_expenditure_region.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 12 - Interactive scatter: Unemployment vs Log GDP (OLS trendline)
df_unemp = df.dropna(subset=["unemployment_rate", "log_gdp_per_capita", "region"])
fig = px.scatter(
    df_unemp,
    x="log_gdp_per_capita",
    y="unemployment_rate",
    color="region",
    opacity=0.4,
    hover_name="country_name",
    hover_data={"year": True},
    trendline="ols",
    labels={
        "log_gdp_per_capita" : "Log GDP per Capita",
        "unemployment_rate"  : "Unemployment Rate (%)",
    },
    title="Unemployment Rate vs Log GDP per Capita - OLS Trendline",
)
fig.write_html(VISUALIZATIONS_DIR /"13_unemployment_vs_gdp.html")
fig.show()

In [ ]:
# Chart 13 - Treemap: Average FDI inflows by region and income group (2010-2022)
df_fdi = (
    df.query("year >= 2010")
      .dropna(subset=["fdi_inflows", "region", "income_group"])
      .query("fdi_inflows > 0")
      .groupby(["region", "income_group"], as_index=False)["fdi_inflows"]
      .mean()
)
fig = px.treemap(
    df_fdi,
    path=["region", "income_group"],
    values="fdi_inflows",
    color="fdi_inflows",
    color_continuous_scale="Blues",
    title="Average FDI Inflows by Region and Income Group (2010-2022)",
    labels={"fdi_inflows": "Avg FDI Inflows (USD)"},
)
fig.write_html(VISUALIZATIONS_DIR /"14_fdi_treemap.html")
fig.show()

In [ ]:
# Chart 14 - Area: Tertiary enrollment trend by income group
df_tert = (
    df.dropna(subset=["tertiary_enrollment", "income_group"])
      .groupby(["income_group", "year"], as_index=False)["tertiary_enrollment"]
      .mean()
)
df_tert["income_group"] = pd.Categorical(
    df_tert["income_group"], categories=INCOME_ORDER, ordered=True
)
fig = px.area(
    df_tert.dropna(subset=["income_group"]),
    x="year", y="tertiary_enrollment",
    color="income_group",
    category_orders={"income_group": INCOME_ORDER},
    labels={
        "tertiary_enrollment": "Tertiary Enrollment (%)",
        "year"               : "Year",
        "income_group"       : "Income Group",
    },
    title="Tertiary School Enrollment Trend by Income Group (2000-2022)",
)
fig.write_html(VISUALIZATIONS_DIR /"15_tertiary_enrollment.html")
fig.show()

In [ ]:
# Section 9 - Statistical Analysis

print("=" * 62)
print("REGIONAL STATISTICS (latest available year per country)")
print("=" * 62)

df_latest_country = (
    df.sort_values("year")
      .groupby("country_code", as_index=False)
      .last()
      .dropna(subset=["region"])
)

regional_stats = (
    df_latest_country
    .groupby("region")[["gdp_per_capita", "life_expectancy",
                         "literacy_rate", "unemployment_rate"]]
    .agg(["mean", "median", "std"])
    .round(2)
)
print(regional_stats)

print("\n" + "=" * 62)
print("INCOME GROUP STATISTICS")
print("=" * 62)
income_stats = (
    df_latest_country
    .groupby("income_group")[["gdp_per_capita", "life_expectancy",
                               "literacy_rate", "secondary_enrollment"]]
    .mean()
    .reindex([i for i in INCOME_ORDER if i in df_latest_country["income_group"].unique()])
    .round(2)
)
print(income_stats)

In [ ]:
# Pair plot - key numeric variables coloured by income group
df_pair = df.dropna(
    subset=["log_gdp_per_capita", "life_expectancy",
            "unemployment_rate", "secondary_enrollment", "income_group"]
).copy()
df_pair["income_group"] = pd.Categorical(
    df_pair["income_group"], categories=INCOME_ORDER, ordered=True
)

pp = sns.pairplot(
    df_pair.dropna(subset=["income_group"]),
    vars=["log_gdp_per_capita", "life_expectancy",
          "unemployment_rate",  "secondary_enrollment"],
    hue="income_group",
    hue_order=INCOME_ORDER,
    plot_kws=dict(alpha=0.3, s=10),
    diag_kind="kde",
    palette="Set2",
)
pp.figure.suptitle("Pair Plot - Key Indicators by Income Group",
                   y=1.01, fontsize=13, fontweight="bold")
pp.figure.savefig(VISUALIZATIONS_DIR /"16_pairplot.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Section 10 - Predictive Modelling (Ridge Regression)

TARGET   = "log_gdp_per_capita"
FEATURES = [
    "life_expectancy", "unemployment_rate",
    "literacy_rate",   "secondary_enrollment",
    "tertiary_enrollment", "edu_expenditure_pct_gdp",
]

df_model = df.dropna(subset=[TARGET] + FEATURES).copy()
print(f"Modelling dataset: {df_model.shape[0]:,} rows, {len(FEATURES)} features")

X = df_model[FEATURES].values
y = df_model[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")

In [ ]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("ridge",   Ridge(alpha=1.0)),
])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
rmse   = float(np.sqrt(mean_squared_error(y_test, y_pred)))
r2     = float(r2_score(y_test, y_pred))
cv_r2  = cross_val_score(pipeline, X, y, cv=5, scoring="r2")

print(f"Test RMSE      : {rmse:.4f}")
print(f"Test R2        : {r2:.4f}")
print(f"CV R2 (5-fold) : {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Actual vs Predicted scatter
axes[0].scatter(y_test, y_pred, alpha=0.35, s=18, color="steelblue")
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, "r--", linewidth=1.5)
axes[0].set_xlabel("Actual Log GDP per Capita")
axes[0].set_ylabel("Predicted Log GDP per Capita")
axes[0].set_title(f"Actual vs Predicted (R2 = {r2:.3f})", fontweight="bold")

# Residual histogram
residuals = y_test - y_pred
axes[1].hist(residuals, bins=40, color="coral", edgecolor="white", alpha=0.85)
axes[1].axvline(0, color="black", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution", fontweight="bold")

plt.suptitle("Ridge Regression Model Diagnostics",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(VISUALIZATIONS_DIR /"17_model_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Feature importance - Ridge coefficients after standardisation
coefs = pd.Series(
    pipeline.named_steps["ridge"].coef_,
    index=FEATURES,
).sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
coefs.plot(
    kind="barh", ax=ax,
    color=["tomato" if v < 0 else "steelblue" for v in coefs],
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Ridge Regression - Feature Coefficients (Standardised)",
             fontweight="bold")
ax.set_xlabel("Coefficient Value")
plt.tight_layout()
plt.savefig(VISUALIZATIONS_DIR /"18_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Section 11 - Insights via Database Queries

print("=" * 62)
print("SQL QUERY 1 - Top 10 countries by average GDP per capita")
print("=" * 62)
q1 = """
    SELECT c.country_name, c.region, c.income_group,
           ROUND(AVG(e.gdp_per_capita)::numeric, 2) AS avg_gdp
    FROM   economic_indicators e
    JOIN   countries c USING (country_code)
    WHERE  e.gdp_per_capita IS NOT NULL
    GROUP  BY c.country_name, c.region, c.income_group
    ORDER  BY avg_gdp DESC
    LIMIT  10;
"""
with pg_engine.connect() as conn:
    df_q1 = pd.read_sql(text(q1), conn)
print(df_q1.to_string(index=False))

In [ ]:
print("=" * 62)
print("SQL QUERY 2 - Avg education expenditure by region")
print("=" * 62)
q2 = """
    SELECT c.region,
           ROUND(AVG(ed.edu_expenditure_pct_gdp)::numeric, 2) AS avg_edu_exp,
           COUNT(DISTINCT ed.country_code) AS countries
    FROM   education_indicators ed
    JOIN   countries c USING (country_code)
    WHERE  ed.edu_expenditure_pct_gdp IS NOT NULL
    GROUP  BY c.region
    ORDER  BY avg_edu_exp DESC;
"""
with pg_engine.connect() as conn:
    df_q2 = pd.read_sql(text(q2), conn)
print(df_q2.to_string(index=False))

In [ ]:
# MongoDB aggregation - global literacy stats for 2019
pipeline_agg = [
    {
        "$match": {
            "indicator_code": "SE.ADT.LITR.ZS",
            "year": 2019,
            "value": {"$ne": None},
        }
    },
    {
        "$group": {
            "_id": None,
            "avg_literacy"  : {"$avg": "$value"},
            "max_literacy"  : {"$max": "$value"},
            "min_literacy"  : {"$min": "$value"},
            "country_count" : {"$sum": 1},
        }
    },
]
result = list(col_raw_edu.aggregate(pipeline_agg))
if result:
    r = result[0]
    print("MongoDB Aggregation - Adult Literacy Rate (2019):")
    print(f"  Countries with data : {r['country_count']}")
    print(f"  Global average      : {r['avg_literacy']:.2f}%%")
    print(f"  Maximum             : {r['max_literacy']:.2f}%%")
    print(f"  Minimum             : {r['min_literacy']:.2f}%%")

In [ ]:
mongo_client.close()
pg_engine.dispose()
print("All database connections closed.")
print("Notebook execution complete.")